In [1]:
import sys
import os

# Add project root to path
sys.path.append(os.path.abspath(".."))

import json
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [2]:
# Load the main.config.json file 

def load_main_config():
    with open("../config/main.config.json", "r") as f:
        return json.load(f)

main_config = load_main_config()
main_config

{'active_version': 'v1',
 'paths': {'model': 'models/v1/model_bundle.pkl',
  'thresholds': 'config/v1/threshold-config.json',
  'data_config': 'config/v1/data-config.json'},
 'data': {'filename': 'synthetic_crudop_logs.csv'},
 's3': {'bucket': 'crud-sla', 'prefix': 'anomaly-detection'},
 'flags': {'upload_to_s3': False, 'download_from_s3': False}}

In [3]:
def get_path(key, config):
    return "../" + config["paths"][key]

data_config_path = get_path("data_config", main_config)

with open(data_config_path, "r") as f:
    data_config = json.load(f)

data_config.keys()

dict_keys(['_meta', 'randomness', 'commands_meta', 'commands', 'hourly_rules_meta', 'hourly_rules'])

In [4]:
np.random.seed(42)

def add_noise(value, pct):
    return value * (1 + np.random.uniform(-pct, pct))

def random_in_range(low, high, pct):
    base = np.random.uniform(low, high)
    return add_noise(base, pct)

def apply_hourly_rules(cmd, hour, values, hourly_rules):
    hour_str = str(hour)

    if hour_str in hourly_rules and cmd in hourly_rules[hour_str]:
        rules = hourly_rules[hour_str][cmd]

        for key, multiplier in rules.items():
            field = key.replace("_multiplier", "")
            if field in values:
                values[field] *= multiplier

    return values

In [5]:
def inject_anomaly(values):

    metric = np.random.choice([
        "success_vol",
        "fail_vol",
        "success_rt",
        "fail_rt"
    ])

    direction = np.random.choice(["up", "down"])

    factor = np.random.uniform(3, 10) if direction == "up" else np.random.uniform(0.1, 0.3)

    values[metric] *= factor

    return values, f"{metric}_{direction}"

In [6]:
def generate_test_data(start_date, hours, config, anomaly_prob=0.2):

    RANDOMNESS = config["randomness"]
    commands_config = config["commands"]
    hourly_rules = config["hourly_rules"]

    data = []
    current = start_date

    for _ in range(hours):
        hour = current.hour

        for cmd, cfg in commands_config.items():

            values = {
                "success_vol": add_noise(cfg["success_vol"], RANDOMNESS),
                "fail_vol": add_noise(cfg["fail_vol"], RANDOMNESS),
                "success_rt": random_in_range(*cfg["success_rt"], RANDOMNESS),
                "fail_rt": random_in_range(*cfg["fail_rt"], RANDOMNESS)
            }

            values = apply_hourly_rules(cmd, hour, values, hourly_rules)

            is_anomaly = False
            anomaly_type = "normal"

            if np.random.rand() < anomaly_prob:
                values, anomaly_type = inject_anomaly(values)
                is_anomaly = True

            data.append({
                "timestamp": current,
                "operation": cmd,   # changed from command → operation
                "success_vol": int(values["success_vol"]),
                "success_rt_avg": round(values["success_rt"], 3),
                "fail_vol": int(values["fail_vol"]),
                "fail_rt_avg": round(values["fail_rt"], 3),
                "is_anomaly": is_anomaly,
                "anomaly_type": anomaly_type
            })

        current += timedelta(hours=1)

    return pd.DataFrame(data)

In [7]:
df_test = generate_test_data(
    start_date=datetime(2025, 1, 1),
    hours=48,   # start with 1 week
    config=data_config
)

df_test.head()

,timestamp,operation,success_vol,success_rt_avg,fail_vol,fail_rt_avg,is_anomaly,anomaly_type
0,2025-01-01 00:00:00,GET,78996,12.441,418,17.177,True,fail_rt_up
1,2025-01-01 00:00:00,POST,10208,40.708,1428,2.406,True,fail_rt_down
2,2025-01-01 00:00:00,PUT,14897,46.612,2937,19.123,False,normal
3,2025-01-01 00:00:00,DELETE,7199,6.643,1745,10.577,True,success_rt_down
4,2025-01-01 01:00:00,GET,8948,7.722,412,4.028,True,success_vol_down


In [8]:
def prepare_features(df):
    df = df.copy()

    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["hour"] = df["timestamp"].dt.hour
    
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    
    # one-hot encoding
    dummies = pd.get_dummies(df["operation"], prefix="operation")
    df = pd.concat([df, dummies], axis=1)
    
    return df

In [9]:
import joblib

bundle = joblib.load("../models/v1/model_bundle.pkl")

models = bundle["models"]
features = bundle["feature_map"]
targets = bundle["targets"]
thresholds = bundle["thresholds"]

In [10]:
def run_inference(df):
    
    df = prepare_features(df)
    results = []

    for _, row in df.iterrows():
        
        op = row["operation"]
        hour = row["hour"]
        success_vol = row["success_vol"]
        fail_vol = row["fail_vol"]
        success_rt_avg = row["success_rt_avg"]
        fail_rt_avg = row["fail_rt_avg"]
        is_anomaly = False
        max_severity = 0
        root_cause = None
        
        for t in targets:
            feat = features[t]
            pred = models[t].predict([row[feat]])[0]
            actual = row[t]
            
            rule = thresholds[op][hour][t]
            
            threshold_val = max(
                pred * rule["percent_threshold"],
                rule["abs_threshold"]
            )
            
            deviation = abs(actual - pred)
            
            if deviation > threshold_val:
                is_anomaly = True
                
                severity = deviation / threshold_val
                
                if severity > max_severity:
                    max_severity = severity
                    root_cause = t
        
        results.append({
            "operation": op,
            "hour": hour,
            "success_vol":success_vol,
            "fail_vol":fail_vol,
            "success_rt_avg":success_rt_avg,
            "fail_rt_avg":fail_rt_avg,
            "Status": "Anomaly 🚨" if is_anomaly else "Normal ✅",
            "Root_Cause": root_cause if is_anomaly else None
        })
    
    return pd.DataFrame(results)

In [11]:
def evaluate(df, results_df):
    
    merged = results_df.copy()
    merged["actual"] = df["is_anomaly"].values
    
    total = len(merged)
    alerts = (merged["Status"] != "Normal ✅").sum()
    actual = merged["actual"].sum()
    
    tp = ((merged["Status"] != "Normal ✅") & merged["actual"]).sum()
    fp = ((merged["Status"] != "Normal ✅") & ~merged["actual"]).sum()
    fn = ((merged["Status"] == "Normal ✅") & merged["actual"]).sum()
    
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)
    alert_rate = alerts / total
    
    print("\n===== PERFORMANCE =====")
    print(f"Total: {total}")
    print(f"Actual Anomalies: {actual}")
    print(f"Detected Alerts: {alerts}")
    
    print(f"\nPrecision: {precision:.3f}")
    print(f"Recall: {recall:.3f}")
    print(f"Alert Rate: {alert_rate:.3f}")

In [14]:
# 1. Generate data
df_test = generate_test_data(start_date=datetime(2025, 1, 1), hours=48, config=data_config, anomaly_prob=0.2)

# 2. Run inference
results = run_inference(df_test)

# 3. Evaluate
evaluate(df_test, results)

# 4. View sample
print(results.head(30))


===== PERFORMANCE =====
Total: 192
Actual Anomalies: 42
Detected Alerts: 47

Precision: 0.872
Recall: 0.976
Alert Rate: 0.245
   operation  hour  success_vol  fail_vol  success_rt_avg  fail_rt_avg  \
0        GET     0        78837       396           6.387        3.208   
1       POST     0         9942      1522          21.895       19.694   
2        PUT     0         1918      2993          48.447       20.682   
3     DELETE     0         7164      1747          15.962       16.546   
4        GET     1        78763       412           9.965        3.848   
5       POST     1         9796      1446          22.874       15.288   
6        PUT     1        15382      3025          52.550       22.441   
7     DELETE     1         7198       190          28.020       19.634   
8        GET     2        79243       400          11.278        4.009   
9       POST     2        10450      1428           8.696       17.745   
10       PUT     2        14821      2866          49.915  